In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_core.language_models.llms import LLM
from typing import Optional, List

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


class TinyLlamaLLM(LLM):
    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    @property
    def _llm_type(self) -> str:
        return "tinyllama_custom"


llm = TinyLlamaLLM()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
import time
import requests

def safe_wikipedia(query: str, retries: int = 2):
    for attempt in range(retries):
        try:
            result = wikipedia_tool.invoke(query)
            return result

        except requests.exceptions.ReadTimeout:
            print(f"Timeout... retrying ({attempt+1})")
            time.sleep(1)

        except Exception as e:
            return f"Error: {str(e)}"

    return "Failed after retries"

In [18]:
print(safe_wikipedia("Alan Turing"))

Page: Alan Turing
Summary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.
Born in London, Turing was raised in southern England. He graduated from King's College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain's codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German ciphers, including improvements to 

Multi Agent

In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_core.language_models.llms import LLM
from typing import Optional, List

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


class TinyLlamaLLM(LLM):
    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    @property
    def _llm_type(self) -> str:
        return "tinyllama_custom"


llm = TinyLlamaLLM()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

In [21]:
from langchain_community.agent_toolkits.load_tools import load_tools
import time
import requests

tools = load_tools(["wikipedia"])
wikipedia_tool = tools[0]


def safe_wikipedia(query: str, retries: int = 2):
    for attempt in range(retries):
        try:
            return wikipedia_tool.invoke(query)

        except requests.exceptions.ReadTimeout:
            print(f"Timeout... retrying ({attempt+1})")
            time.sleep(1)

        except Exception as e:
            return f"Error: {str(e)}"

    return "Failed after retries"

In [22]:
# Math Agent
def math_agent(query: str):
    return calculator.invoke(query)


# Wiki Agent
def wiki_agent(query: str):
    return safe_wikipedia(query)


# LLM Agent (fallback)
def llm_agent(query: str):
    return llm.invoke(query)

In [23]:
def route_query(query: str):
    q = query.lower()

    if any(op in q for op in ["+", "-", "*", "/"]):
        return "math"

    elif any(word in q for word in ["who", "what", "when", "where"]):
        return "wiki"

    else:
        return "llm"

In [24]:
def multi_agent_system(query: str):
    route = route_query(query)

    if route == "math":
        return {
            "agent": "math_agent",
            "output": math_agent(query)
        }

    elif route == "wiki":
        return {
            "agent": "wiki_agent",
            "output": wiki_agent(query)
        }

    else:
        return {
            "agent": "llm_agent",
            "output": llm_agent(query)
        }

In [25]:
print(multi_agent_system("15 * 6"))
print(multi_agent_system("Who is Alan Turing?"))
print(multi_agent_system("Explain embeddings in simple terms"))

{'agent': 'math_agent', 'output': '90'}
Timeout... retrying (1)


Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'agent': 'wiki_agent', 'output': 'Page: Turing Award\nSummary: The ACM A. M. Turing Award is an annual prize given by the Association for Computing Machinery (ACM) for contributions of lasting and major technical importance to computer science. It is generally recognized as the highest distinction in the field of computer science and is often referred to as the "Nobel Prize of Computing". As of 2025, 79 people have been awarded the prize, with the most recent recipients being Andrew Barto and Richard S. Sutton, who won in 2024.\nThe award is named after Alan Turing, also referred as "Father of Computer Science", who was a British mathematician and reader in mathematics at the University of Manchester. Turing is often credited as being the founder of theoretical computer science and artificial intelligence, and a key contributor to the Allied cryptanalysis of the Enigma cipher during World War II. From 2007 to 2013, the award was accompanied by a prize of US$250,000, with financial sup

In [26]:
# ============================================================
# DATAFRAME AGENT WITH LARGE HF MODEL (COMMENTED TEMPLATE)
# ============================================================

# -------------------------------
# 1. INSTALL DEPENDENCIES
# -------------------------------
# !pip install -q pandas langchain langchain-community langchain-experimental transformers accelerate torch tabulate


# -------------------------------
# 2. LOAD LARGE MODEL (MISTRAL 7B)
# -------------------------------
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from langchain_core.language_models.llms import LLM
# from typing import Optional, List

# device = "cuda" if torch.cuda.is_available() else "cpu"

# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

# model = AutoModelForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-Instruct-v0.1",
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32
# ).to(device)


# -------------------------------
# 3. CUSTOM LLM WRAPPER
# -------------------------------
# class MistralLLM(LLM):
#     def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
#         inputs = tokenizer(prompt, return_tensors="pt").to(device)

#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=256,
#             do_sample=False
#         )

#         return tokenizer.decode(outputs[0], skip_special_tokens=True)

#     @property
#     def _llm_type(self) -> str:
#         return "mistral_custom"

# llm = MistralLLM()


# -------------------------------
# 4. CREATE DATAFRAME
# -------------------------------
# import pandas as pd

# data = {
#     "name": ["Alice", "Bob", "Charlie", "David"],
#     "age": [25, 30, 35, 40],
#     "salary": [50000, 60000, 70000, 80000]
# }

# df = pd.DataFrame(data)
# print(df)


# -------------------------------
# 5. CREATE DATAFRAME AGENT
# -------------------------------
# from langchain_experimental.agents import create_pandas_dataframe_agent

# df_agent = create_pandas_dataframe_agent(
#     llm,
#     df,
#     verbose=True,
#     allow_dangerous_code=True
# )


# -------------------------------
# 6. RUN QUERIES
# -------------------------------
# print(df_agent.invoke({"input": "What is the average salary?"}))
# print(df_agent.invoke({"input": "Who has the highest salary?"}))
# print(df_agent.invoke({"input": "How many people are older than 30?"}))


# ============================================================
# NOTES:
# - Requires ~14GB VRAM for Mistral 7B (fp16)
# - Will NOT run on low-memory GPUs
# - Use smaller models if needed:
#     "microsoft/phi-2"
#     "google/gemma-2b-it"
# ============================================================

In [27]:
# ============================================================
# MULTI-DATAFRAME AGENT (COMMENTED TEMPLATE)
# ============================================================

# -------------------------------
# 1. INSTALL DEPENDENCIES
# -------------------------------
# !pip install -q pandas langchain langchain-community langchain-experimental transformers accelerate torch tabulate


# -------------------------------
# 2. LOAD LARGE MODEL (MISTRAL 7B)
# -------------------------------
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from langchain_core.language_models.llms import LLM
# from typing import Optional, List

# device = "cuda" if torch.cuda.is_available() else "cpu"

# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

# model = AutoModelForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-Instruct-v0.1",
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32
# ).to(device)


# -------------------------------
# 3. CUSTOM LLM WRAPPER
# -------------------------------
# class MistralLLM(LLM):
#     def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
#         inputs = tokenizer(prompt, return_tensors="pt").to(device)

#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=256,
#             do_sample=False
#         )

#         return tokenizer.decode(outputs[0], skip_special_tokens=True)

#     @property
#     def _llm_type(self) -> str:
#         return "mistral_custom"

# llm = MistralLLM()


# -------------------------------
# 4. CREATE MULTIPLE DATAFRAMES
# -------------------------------
# import pandas as pd

# # Employee Data
# df_employees = pd.DataFrame({
#     "emp_id": [1, 2, 3, 4],
#     "name": ["Alice", "Bob", "Charlie", "David"],
#     "dept_id": [101, 102, 101, 103],
#     "salary": [50000, 60000, 70000, 80000]
# })

# # Department Data
# df_departments = pd.DataFrame({
#     "dept_id": [101, 102, 103],
#     "department": ["HR", "Engineering", "Marketing"]
# })

# print(df_employees)
# print(df_departments)


# -------------------------------
# 5. CREATE MULTI-DF AGENT
# -------------------------------
# from langchain_experimental.agents import create_pandas_dataframe_agent

# df_agent = create_pandas_dataframe_agent(
#     llm,
#     [df_employees, df_departments],   # 🔥 multiple dataframes
#     verbose=True,
#     allow_dangerous_code=True
# )


# -------------------------------
# 6. RUN CROSS-DATAFRAME QUERIES
# -------------------------------
# # Join-like query
# print(df_agent.invoke({
#     "input": "Which department does Alice work in?"
# }))

# # Aggregation across join
# print(df_agent.invoke({
#     "input": "What is the average salary in each department?"
# }))

# # Filtering + join
# print(df_agent.invoke({
#     "input": "List employees in Engineering with salary > 55000"
# }))


# ============================================================
# NOTES:
# - LLM internally figures out joins (merge operations)
# - Works best with strong models (Mistral, GPT-4, etc.)
# - Small models will fail at joins / reasoning
# - Requires high VRAM (~14GB for Mistral 7B)
# ============================================================